# 5. Validação de ciclo

Mesma checagem de `src/lib/craftingGraph.ts` (DFS de 3 estados), rodada
aqui sobre o grafo bruto de recipes antes do export - o objetivo é falhar
cedo no pipeline de dados, em vez de só descobrir um ciclo em runtime no app.


In [1]:
import json
from pathlib import Path

import pandas as pd

# Notebook lives in notebooks/etl/, so the repo root is two levels up.
REPO_ROOT = Path("../..").resolve()

ROOT = REPO_ROOT / "data/Pal/Content"
PAL = ROOT / "Pal"
ITEM_DT = PAL / "DataTable/Item/DT_ItemDataTable_Common.json"
RECIPE_DT = PAL / "DataTable/Item/DT_ItemRecipeDataTable_Common.json"
BUILDOBJECT_DT = PAL / "DataTable/MapObject/Building/DT_BuildObjectDataTable_Common.json"
BENCH_RECIPES = REPO_ROOT / "src/data/bench_recipes.json"
NAMES_DT_EN = ROOT / "L10N/en/Pal/DataTable/Text/DT_ItemNameText_Common.json"
NAMES_DT_PT_BR = ROOT / "L10N/pt-BR/Pal/DataTable/Text/DT_ItemNameText_Common.json"


def load_rows(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)[0]["Rows"]


In [2]:
items_raw = load_rows(ITEM_DT)
recipes_raw = load_rows(RECIPE_DT)
buildings_raw = load_rows(BUILDOBJECT_DT)

items_df = pd.DataFrame.from_dict(items_raw, orient="index")
recipes_df = pd.DataFrame.from_dict(recipes_raw, orient="index")
buildings_df = pd.DataFrame.from_dict(buildings_raw, orient="index")

print(f"items: {items_df.shape}, recipes: {recipes_df.shape}, buildings: {buildings_df.shape}")


items: (2466, 53), recipes: (1414, 20), buildings: (498, 32)


In [3]:
# Padroniza a string sentinela "None" (usada pela UE pra ausência de valor)
# para o None real do Python, e monta um resolvedor de id case-insensitive -
# ver 02_limpeza.ipynb pro raciocínio completo por trás disso.
items_clean = items_df.replace("None", None)
recipes_clean = recipes_df.replace("None", None)
buildings_clean = buildings_df.replace("None", None)

items_by_lower = {item_id.lower(): item_id for item_id in items_clean.index}


def resolve_item_id(raw_id):
    # pandas' .replace("None", None) upcasts these cells to NaN (a float),
    # not Python None - "is None" or plain truthiness checks silently miss
    # it and .lower() blows up on a float. pd.isna() catches both.
    if pd.isna(raw_id):
        return None
    if raw_id in items_clean.index:
        return raw_id
    return items_by_lower.get(raw_id.lower())


## Montar o grafo de dependências (item -> ingredientes)


In [4]:
graph = {}
for product_id, recipe in recipes_clean.iterrows():
    deps = []
    for i in range(1, 6):
        material_id = recipe[f"Material{i}_Id"]
        material_count = recipe[f"Material{i}_Count"] or 0
        if pd.isna(material_id) or material_count == 0:
            continue
        resolved = resolve_item_id(material_id)
        if resolved:
            deps.append(resolved)
    graph[product_id] = deps

print(f"{len(graph)} nós com receita no grafo")


1414 nós com receita no grafo


## DFS de 3 estados (unvisited / in_progress / done)


In [5]:
UNVISITED, IN_PROGRESS, DONE = 0, 1, 2


def find_cycles(graph):
    state = {node: UNVISITED for node in graph}
    cycles = []

    def visit(node, path):
        state[node] = IN_PROGRESS
        path.append(node)
        for dep in graph.get(node, []):
            dep_state = state.get(dep, DONE)
            if dep_state == IN_PROGRESS:
                cycle_start = path.index(dep)
                cycles.append(path[cycle_start:] + [dep])
            elif dep_state == UNVISITED:
                visit(dep, path)
        path.pop()
        state[node] = DONE

    for node in graph:
        if state[node] == UNVISITED:
            visit(node, [])
    return cycles


cycles = find_cycles(graph)
print(f"{len(cycles)} ciclo(s) encontrado(s) no grafo real")
cycles


0 ciclo(s) encontrado(s) no grafo real


[]

## Controle positivo

Prova que o detector realmente pega um ciclo, injetando um sintético -
sem isso, "0 ciclos encontrados" acima poderia só significar um detector
quebrado.


In [6]:
test_graph = {"A": ["B"], "B": ["C"], "C": ["A"]}
test_cycles = find_cycles(test_graph)
assert test_cycles, "detector de ciclo não pegou um ciclo sintético óbvio - tem bug no detector"
print("Controle positivo OK:", test_cycles)


Controle positivo OK: [['A', 'B', 'C', 'A']]
